# Objetivo

Realizar a transformação e modelagem dos dados, adequando os DataFrames ao modelo lógico definido para posterior carregamento em um banco de dados.

* Criar novas colunas derivadas;
* Criar uma dimensão de datas;
* Renomear e organizar as colunas;
* Preparar as chaves utilizadas no modelo;
* Substituir a identificação composta de Order Items por uma chave substituta.

# Importações

In [ ]:
from pipeline_vendas.extract import extract_processed_data
from pipeline_vendas.transform import transform_data
from pipeline_vendas.transform import format_data
from pipeline_vendas.validation import validate_data
from pipeline_vendas.load import run_load

# Exibição da validação

In [ ]:
def display_validation(validation: dict) -> None:
    """
    Exibe os resultados das validações de forma organizada.
    
    Parameters
    ----------
    validation : dict
        Dicionário contendo os resultados das validações
    """

    print("===== QUANTIDADE DE REGISTROS =====")

    for name, (before, after) in validation["row_counts"].items():
        print(f"{name}: {before} -> {after}")

    print("\n===== CHAVES PRIMÁRIAS =====")

    for name, result in validation["primary_keys"].items():
        status = "OK" if result else "ERRO"
        print(f"{name}: {status}")

    print("\n===== CHAVES ESTRANGEIRAS =====")

    for name, result in validation["foreign_keys"].items():
        status = "OK" if result else "ERRO"
        print(f"{name}: {status}")

# Extração dos Dados

In [ ]:
processed_dfs = extract_processed_data()

# Transformação e Modelagem

In [ ]:
transformed_dfs = transform_data(processed_dfs)
formatted_dfs = format_data(transformed_dfs)

# Validação

In [ ]:
validation = validate_data(processed_dfs, transformed_dfs, formatted_dfs)

display_validation(validation)

# Exportação dos Dados

In [ ]:
run_load(formatted_dfs)

# Conclusão

Nesta etapa, os dados processados foram transformados e modelados de acordo com o modelo lógico definido para o projeto, preparando os DataFrames para posterior carregamento no banco de dados.

Foram estruturadas as seguintes tabelas:

* `dim_customers`: dimensão de clientes, com um registro por cliente identificado por `customer_id`;
* `dim_products`: dimensão de produtos, com um registro por produto;
* `dim_sellers`: dimensão de vendedores, com um registro por vendedor;
* `dim_date`: dimensão de datas, contendo atributos derivados da data de compra;
* `fact_orders`: tabela fato com granularidade de um registro por pedido;
* `fact_order_items`: tabela fato com granularidade de um registro por item de pedido.

Na tabela `fact_order_items`, foi criada a coluna `item_id` como chave substituta, permitindo identificar unicamente cada registro sem depender da identificação composta utilizada originalmente no dataset.

Também foram criadas colunas derivadas, como `total_value`, correspondente à soma do preço do item e do frete, e `purchase_date`, utilizada para relacionar os itens à dimensão de datas.

### Relacionamento entre as tabelas

* `Customers (1) ----> Orders (0, N)`
* `Orders (1) ----> Order Items (1, N)`
* `Products (1) ----> Order Items (1, N)`
* `Sellers (1) ----> Order Items (1, N)`
* `Date (1) ----> Order Items (1, N)`

O relacionamento entre `Customers` e `Orders` permite identificar os pedidos associados a cada cliente. `Order Items` representa os itens pertencentes a cada pedido, relacionando cada registro ao respectivo produto, vendedor e data da compra.

As transformações realizadas nesta etapa adequam os DataFrames ao modelo lógico definido, mantendo a granularidade das tabelas e preparando as colunas que serão utilizadas como chaves primárias e estrangeiras no banco de dados.

Também foi realizada uma etapa de validação dos dados transformados, verificando a quantidade de registros, a unicidade das chaves primárias e a integridade das chaves estrangeiras antes do carregamento no banco de dados.

Abaixo, é apresentado o modelo lógico final dos DataFrames resultantes desta etapa.

![Modelo Lógico Final](../reports/images/Modelo_Logico_Final.jpeg)